# 03 · Sızıntı kontrolü ve geçerlilik maskesi (Adım 7)

İki yerde çalışır: **yerelde (VS Code)** ya da Kaggle. İki iş: (1) event'ler arası coğrafi örtüşme taraması (`src.leak_check`, eşikler koddan ÖNCE yazıldı), (2) öncesi/sonrası boş bölgeler için değerlendirme geçerlilik maskesi (`src.tile_validity`).

**Yerelde:** defteri `deprem-vlm-rag/notebooks/` içinden aç. Veri `kagglehub` ile Kaggle dataset'inden `~/.cache/kagglehub` altına (repo DIŞI, ~5 GB, SAR dahil) iner; çıktılar `RAGEarthQuake/ciktilar/adim7_sizinti_gecerlilik/`. Kaggle'da: `kod` + `melihkotman/disasterm3-turkiye` girdileri, Accelerator: None.

In [ ]:
import os, sys, glob, shutil, pathlib, subprocess
KAGGLE = os.path.exists("/kaggle/input")
if KAGGLE:      # Kaggle: kod, Add Input ile eklenen 'kod' dataset'inden (GitHub/Secrets gerekmez)
    hit = glob.glob("/kaggle/input/**/src/leak_check.py", recursive=True)
    if not hit: raise SystemExit("'kod' dataset'i ekli değil: Add Input -> kod")
    if os.path.exists("/tmp/repo"): shutil.rmtree("/tmp/repo")
    shutil.copytree(os.path.dirname(os.path.dirname(hit[0])), "/tmp/repo")
    REPO, DATA, OUT = pathlib.Path("/tmp/repo"), None, pathlib.Path("/kaggle/working")
else:           # Yerel (VS Code): defter deprem-vlm-rag/notebooks içinden açılmalı
    p = pathlib.Path.cwd().resolve()
    while p != p.parent and not (p / "src" / "leak_check.py").exists(): p = p.parent
    if not (p / "src" / "leak_check.py").exists(): raise SystemExit("repo kökü bulunamadı: defteri deprem-vlm-rag/notebooks içinden aç")
    REPO = p
    DATA = pathlib.Path(os.environ.get("RAG_DATA", pathlib.Path.home() / "DM3Veri" / "dm3_turkiye"))   # repo DIŞINDA
    OUT = REPO.parent / "ciktilar" / "adim7_sizinti_gecerlilik"                                                                 # repo DIŞINDA
    os.environ["RAG_DATA"] = str(DATA)
OUT.mkdir(parents=True, exist_ok=True)
os.chdir(REPO)
def run(*args):
    print("$ python", *args, flush=True)
    r = subprocess.run([sys.executable, *args], cwd=REPO)
    if r.returncode: print("-> çıkış kodu", r.returncode)
    return r.returncode
print("kip:", "Kaggle" if KAGGLE else "yerel", "| repo:", REPO, "| veri:", DATA or "(Kaggle girdisi)", "| çıktı:", OUT)
run("-m", "pip", "-q", "install", "-r", "requirements.txt")
run("-m", "src.config")

## 0. Veri: `kagglehub` ile Kaggle dataset'inden (Adım 5'te doğrulanmış kopya). Önbellek `~/.cache/kagglehub` altına iner, ayrıca kopyalanmaz

In [ ]:
if not KAGGLE:
    run("-m", "pip", "-q", "install", "kagglehub")
    import kagglehub
    try:
        path = kagglehub.dataset_download("melihkotman/disasterm3-turkiye")
    except Exception as e:                   # özel dataset: giriş gerekir (kullanıcı adı + API key/token sorar)
        print("kimlik doğrulama gerekiyor:", type(e).__name__)
        kagglehub.login()
        path = kagglehub.dataset_download("melihkotman/disasterm3-turkiye")
    print("Path to dataset files:", path)
    # bench/ ve train/ içeren klasörü bul (dm3_turkiye/ altında olabilir)
    DATA = next((pathlib.Path(dp) for dp, dn, _ in os.walk(path) if "bench" in dn and "train" in dn), None)
    if DATA is None: raise SystemExit("bench/ ve train/ bulunamadı: " + str(path))
    os.environ["RAG_DATA"] = str(DATA)
    print("veri kökü:", DATA)
run("-m", "src.dm3_verify")

`SONUÇ: OK` ve 1.030 karo görmelisin (SAR'lı karo sayısı yerelde 0 olabilir, sorun değil).

## 1. Deneme (her event'ten 8 karo)

In [ ]:
run("-m", "src.leak_check", "--out", str(OUT / "leak_deneme"), "--limit", "8")

`pozitif kontrol (kırpık kopya, KAPI)` ≥ 0.80 olmalı (önceki deneme: ≈0,95). Değilse DUR, çıktıyı gönder.

## 2. Tam koşu (öncesi + sonrası)

In [ ]:
run("-m", "src.leak_check", "--out", str(OUT / "leak"), "--also-post")

In [ ]:
print(open(OUT / "leak" / "ozet.txt").read()); print(sorted(os.listdir(OUT / "leak" / "gorseller"))[:50])

## 3. Geçerlilik maskesi

In [ ]:
run("-m", "src.tile_validity", "--out", str(OUT / "validity"), "--save-masks")
import zipfile
z = OUT / "leak_sonuc.zip"
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in list((OUT / "leak").rglob("*")) + [OUT / "validity" / "ozet.csv", OUT / "validity" / "tile_validity_v1.csv"]:
        if f.is_file(): zf.write(f, f.relative_to(OUT))
print(z, round(z.stat().st_size / 1e6, 1), "MB")

Bitince `leak_sonuc.zip`'i (içinde `leak/ozet.txt`, `event_matrisi.csv`, `gorseller/`, `validity/ozet.csv`, `tile_validity_v1.csv`) sohbete yükle ya da `ozet.txt` + `validity/ozet.csv` içeriğini yapıştır.